In [ ]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random


In [ ]:
config = dict(
    total_timestep=200_000,
    buffer_size=10_000,
    batch_size=64,
    learning_rate=0.0005,
    eps_start=1,
    eps_end=0.03,
    eps_decay_steps=10_000,
    gamma=0.99,
    target_update_frequency=1000,
    learning_starts=10_000,
    train_update_frequency=10,
    torch_deterministic=True,
    num_envs=1,
    # ---- 평가 / 다중 seed 설정 ----
    eval_interval=5_000,      # 이 step마다 greedy 평가
    eval_episodes=10,         # 평가 1회당 에피소드 수
    seeds=list(range(10)),    # 돌릴 seed 목록
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
'''
class QNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(4,128)
    self.fc2 = nn.Linear(128,4)
    self.fc3 = nn.Linear(4,2)

  def forward(self,a):
    a = self.fc1(a)
    a = torch.relu(a)
    a = self.fc2(a)
    a = torch.relu(a)
    a = self.fc3(a)
    return a
'''
class QNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(4,128)
    self.fc2 = nn.Linear(128,64)
    self.fc3 = nn.Linear(64,2)

  def forward(self,a):
    a = self.fc1(a)
    a = torch.relu(a)
    a = self.fc2(a)
    a = torch.relu(a)
    a = self.fc3(a)
    return a


###Replay buffer

transition저장을 해야함.
transition의 구성요소 -> observation,action,reward,next observation

cartpole transition -> 4차원 실수, 0또는1, 0에서 500사이 실수, 4차원 실수

Replay buffer는 각 구성요소에 대해서 각각의 버퍼를 만드는 것이 좋아보임
버퍼는 원형큐로 만들면됨. 사이즈가 고정되어있기 때문에 그리고 큐가 꽉 찼을 때 가장 오래된 transition부터 제거할 것이기 때문에 원형큐가 유리할 것이라 판단.


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, obs_dim):
        self.capacity = capacity
        self.obs_dim = obs_dim
        self.obs_buffer      = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions_buffer  = np.zeros((capacity, 1), dtype=np.int64)
        self.rewards_buffer  = np.zeros((capacity, 1), dtype=np.float32)
        self.next_obs_buffer = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.dones_buffer    = np.zeros((capacity, 1), dtype=np.float32)
        self.size = 0
        self.ptr = 0

    def add(self, obs, action, reward, next_obs, done):
        obs      = np.asarray(obs, dtype=np.float32).reshape(-1, self.obs_dim)
        next_obs = np.asarray(next_obs, dtype=np.float32).reshape(-1, self.obs_dim)
        action   = np.asarray(action, dtype=np.int64).reshape(-1, 1)
        reward   = np.asarray(reward, dtype=np.float32).reshape(-1, 1)
        done     = np.asarray(done, dtype=np.float32).reshape(-1, 1)

        n = len(obs)
        if n > self.capacity:
            obs, action, reward, next_obs, done = (
                x[-self.capacity:] for x in (obs, action, reward, next_obs, done)
            )
            n = self.capacity

        idx = (self.ptr + np.arange(n)) % self.capacity

        self.obs_buffer[idx]      = obs
        self.actions_buffer[idx]  = action
        self.rewards_buffer[idx]  = reward
        self.next_obs_buffer[idx] = next_obs
        self.dones_buffer[idx]    = done

        self.ptr = (self.ptr + n) % self.capacity
        self.size = min(self.size + n, self.capacity)

    def sample(self, batch_size):
        idx = np.random.randint(0, self.size, size=batch_size)
        return (
            torch.as_tensor(self.obs_buffer[idx]),
            torch.as_tensor(self.actions_buffer[idx]),
            torch.as_tensor(self.rewards_buffer[idx]),
            torch.as_tensor(self.next_obs_buffer[idx]),
            torch.as_tensor(self.dones_buffer[idx]),
        )

    def __len__(self):
        return self.size

In [ ]:
#ε-decay

def epsilon_schedule(step,config):
    frac = min(step / config["eps_decay_steps"], 1.0)
    return config["eps_start"] + frac * (config["eps_end"] - config["eps_start"])

In [ ]:
# ---- 저장 경로 설정 ----
# 결과/체크포인트를 저장할 기본 폴더. 본인 환경에 맞게 바꿔서 사용하세요.
# Colab에서 Google Drive에 저장하려면 아래 두 줄의 주석을 풀고 BASE_DIR을 Drive 안의 폴더로 지정하면 됩니다.
# from google.colab import drive
# drive.mount('/content/drive')
BASE_DIR = './DQN'


In [ ]:
import os
import time

save_dir = os.path.join(BASE_DIR, 'dqn_checkpoints_128-64_huber')
results_dir = os.path.join(BASE_DIR, 'dqn_results_128-64_huber')
os.makedirs(save_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
log_interval = 1000


def make_env(env_id, seed):
    def thunk():
        env = gym.make(env_id)
        env.action_space.seed(seed)
        return env
    return thunk


def evaluate(q_net, n_episodes, seed):
    """greedy 정책으로 n_episodes 돌려서 평균 return 반환"""
    env = gym.make('CartPole-v1')
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        done, ep_ret = False, 0.0
        while not done:
            with torch.no_grad():
                q = q_net(torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0))
            obs, r, term, trunc, _ = env.step(int(q.argmax(dim=1).item()))
            ep_ret += r
            done = term or trunc
        returns.append(ep_ret)
    env.close()
    return float(np.mean(returns))


def train(seed, config):
    # ---------- seed 고정 (네트워크 생성 전에) ----------
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = config['torch_deterministic']

    envs = gym.vector.SyncVectorEnv(
        [make_env('CartPole-v1', seed=seed + i) for i in range(config['num_envs'])]
    )
    rb = ReplayBuffer(config['buffer_size'], envs.single_observation_space.shape[0])
    q_net = QNetwork().to(device)
    target_q_net = QNetwork().to(device)
    target_q_net.load_state_dict(q_net.state_dict())
    optimizer = optim.Adam(q_net.parameters(), lr=config['learning_rate'])

    obs, _ = envs.reset(seed=seed)
    autoreset = np.zeros(config['num_envs'], dtype=bool)
    ep_returns = np.zeros(config['num_envs'])
    episode_returns = []
    eval_steps, eval_returns = [], []
    last_loss = float('nan')
    start_time = time.time()

    def do_eval(step):
        ret = evaluate(q_net, config['eval_episodes'], seed=seed * 100_000 + step)
        eval_steps.append(step)
        eval_returns.append(ret)
        return ret

    try:
        for global_timestep in range(config['total_timestep']):
            env_steps = global_timestep * config['num_envs']

            # 주기적 평가 (step 0 포함)
            if global_timestep % config['eval_interval'] == 0:
                do_eval(env_steps)

            # 행동 선택
            epsilon = epsilon_schedule(global_timestep, config)
            if random.random() < epsilon:
                actions = np.array([envs.single_action_space.sample() for _ in range(config['num_envs'])])
            else:
                with torch.no_grad():
                    q_values = q_net(torch.as_tensor(obs, dtype=torch.float32, device=device))
                    actions = torch.argmax(q_values, dim=1).cpu().numpy()

            next_obs, rewards, terminated, truncated, info = envs.step(actions)

            valid = ~autoreset
            if valid.any():
                rb.add(obs[valid], actions[valid], rewards[valid], next_obs[valid], terminated[valid])

            ep_returns[valid] += rewards[valid]
            done = terminated | truncated
            for i in np.where(done)[0]:
                episode_returns.append(ep_returns[i])
                ep_returns[i] = 0

            autoreset = done
            obs = next_obs

            # 학습
            if global_timestep >= config['learning_starts']:
                if global_timestep % config['train_update_frequency'] == 0:
                    obs_b, act_b, rew_b, next_obs_b, done_b = (t.to(device) for t in rb.sample(config['batch_size']))
                    with torch.no_grad():
                        next_q_max = target_q_net(next_obs_b).max(dim=1, keepdim=True).values
                        target_q_values = rew_b + config['gamma'] * next_q_max * (1 - done_b)
                    q_values = q_net(obs_b).gather(1, act_b)
                    loss = F.smooth_l1_loss(q_values, target_q_values)   # MSE 대신 Huber
                    optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
                    optimizer.step()
                    if global_timestep % log_interval == 0:
                        last_loss = loss.item()

                if global_timestep % config['target_update_frequency'] == 0:
                    target_q_net.load_state_dict(q_net.state_dict())

            if global_timestep % 10_000 == 0 and global_timestep > 0:
                sps = int(env_steps / (time.time() - start_time))
                print(f"[seed {seed}] step {env_steps:>7d} | eval return {eval_returns[-1]:6.1f} "
                      f"| eps {epsilon:.3f} | loss {last_loss:.4f} | SPS {sps}")

        # 마지막 평가 + 최종 모델 저장
        do_eval(config['total_timestep'] * config['num_envs'])
        torch.save({'q_net': q_net.state_dict(), 'config': config, 'seed': seed},
                   os.path.join(save_dir, f'final_seed{seed}.pt'))
    finally:
        envs.close()

    return np.array(eval_steps), np.array(eval_returns)

In [ ]:
for seed in config['seeds']:
    path = os.path.join(results_dir, f'seed{seed}.npz')
    if os.path.exists(path):
        print(f"seed {seed}: 이미 있음, 건너뜀")
        continue
    steps, rets = train(seed, config)
    np.savez(path, steps=steps, returns=rets)
    print(f"seed {seed} 완료 | 최종 eval return {rets[-1]:.1f}")

In [ ]:
import glob
import matplotlib.pyplot as plt
from scipy.stats import trim_mean

def iqm_ci(x, reps=10_000, alpha=0.05, seed=0):
    """x: (num_seeds, ...) -> (IQM, CI 하한, CI 상한). seed 축으로 bootstrap"""
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    n = x.shape[0]
    idx = rng.integers(0, n, size=(reps, n))
    boot = trim_mean(x[idx], 0.25, axis=1)
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)], axis=0)
    return trim_mean(x, 0.25, axis=0), lo, hi

files = sorted(glob.glob(os.path.join(results_dir, 'seed*.npz')))
data = [np.load(f) for f in files]
steps = data[0]['steps']
R = np.stack([d['returns'] for d in data])          # (n_seeds, n_eval_points)
n_seeds = R.shape[0]
print(f"seed {n_seeds}개 로드")

# ---------- 1. 학습 곡선 ----------
m, lo, hi = iqm_ci(R)
fig, ax = plt.subplots(figsize=(7, 4))
for r in R:                                          # 개별 seed (흐리게)
    ax.plot(steps, r, color='gray', alpha=0.2, lw=0.8)
ax.plot(steps, m, lw=2, label=f'DQN IQM (n={n_seeds} seeds)')
ax.fill_between(steps, lo, hi, alpha=0.25, label='95% bootstrap CI')
ax.axhline(475, ls='--', c='k', lw=0.8, label='threshold 475')
ax.set_xlabel('Environment steps')
ax.set_ylabel('Evaluation return (greedy)')
ax.set_ylim(0, 520)
ax.legend(loc='lower right')
ax.set_title('CartPole-v1 DQN')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'learning_curve.png'), dpi=200)
plt.show()

# ---------- 2. seed별 요약 지표 ----------
T = config['total_timestep']
avg_return = np.trapezoid(R, steps, axis=1) / (steps[-1] - steps[0])   # 정규화 AUC = 학습 중 평균 성능
final_return = R[:, -3:].mean(axis=1)                                   # 마지막 3개 평가 지점 평균
solved = R >= 475
reached = solved.any(axis=1)
steps_to_475 = np.where(reached, steps[solved.argmax(axis=1)], T)       # 도달 못 하면 T로 처리

for name, v in [('Normalized AUC', avg_return),
                ('Final return (last 3 evals)', final_return),
                ('Steps to 475', steps_to_475)]:
    mm, l, h = iqm_ci(v)
    print(f"{name:28s} IQM = {mm:9.1f}   95% CI = [{l:9.1f}, {h:9.1f}]")
print(f"475 도달한 seed: {reached.sum()}/{n_seeds}")

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import trim_mean

base = BASE_DIR   # 경로 설정 셀에서 지정한 폴더

# 비교할 설정: {그래프 라벨: 결과 폴더}
runs = {
    'Small':  f'{base}/dqn_results_8-6-4',
    'Bottleneck': f'{base}/dqn_results_128-4',   # 실제 폴더 이름으로 바꾸세요
    'Base' : f'{base}/dqn_results_128-64',
    'Base + Huber + clip' : f'{base}/dqn_results_128-64_huber',
}

def iqm_ci(x, reps=10_000, alpha=0.05, seed=0):
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    n = x.shape[0]
    idx = rng.integers(0, n, size=(reps, n))
    boot = trim_mean(x[idx], 0.25, axis=1)
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)], axis=0)
    return trim_mean(x, 0.25, axis=0), lo, hi

def load_results(folder):
    files = sorted(glob.glob(os.path.join(folder, 'seed*.npz')))
    data = [np.load(f) for f in files]
    steps = data[0]['steps']
    R = np.stack([d['returns'] for d in data])      # (n_seeds, n_points)
    return steps, R

# ---------- 학습 곡선 ----------
fig, ax = plt.subplots(figsize=(7, 4))
results = {}
for label, folder in runs.items():
    steps, R = load_results(folder)
    results[label] = (steps, R)
    m, lo, hi = iqm_ci(R)
    ax.plot(steps, m, lw=2, label=f'{label} (n={len(R)})')
    ax.fill_between(steps, lo, hi, alpha=0.2)

ax.axhline(475, ls='--', c='k', lw=0.8)
ax.set_xlabel('Environment steps')
ax.set_ylabel('IQM evaluation return (95% CI)')
ax.set_ylim(0, 520)
ax.legend(loc='upper left')
ax.set_title('CartPole-v1')
plt.tight_layout()
plt.savefig(f'{base}/learning_curve_compare.png', dpi=200)
plt.show()

# ---------- 요약 지표 ----------
for label, (steps, R) in results.items():
    T = steps[-1]
    auc = np.trapezoid(R, steps, axis=1) / (steps[-1] - steps[0])
    final = R[:, -3:].mean(axis=1)
    solved = R >= 475
    reached = solved.any(axis=1)
    to475 = np.where(reached, steps[solved.argmax(axis=1)], T)

    print(f"=== {label} ===")
    for name, v in [('Normalized AUC', auc), ('Final return', final), ('Steps to 475', to475)]:
        mm, l, h = iqm_ci(v)
        print(f"  {name:15s} IQM = {mm:9.1f}   95% CI = [{l:9.1f}, {h:9.1f}]")
    print(f"  475 도달: {reached.sum()}/{len(R)}\n")

In [ ]:
labels = list(results.keys())
fig, axes = plt.subplots(1, len(labels), figsize=(4 * len(labels), 3.5), sharey=True)

curves = {lb: iqm_ci(R) for lb, (steps, R) in results.items()}
for ax, lb in zip(axes, labels):
    steps, R = results[lb]
    for other in labels:                       # 다른 설정은 회색 참고선
        if other != lb:
            ax.plot(results[other][0], curves[other][0], color='gray', alpha=0.4, lw=1)
    m, lo, hi = curves[lb]
    ax.plot(steps, m, lw=2, color='C0')
    ax.fill_between(steps, lo, hi, alpha=0.25, color='C0')
    ax.axhline(475, ls='--', c='k', lw=0.8)
    ax.set_title(f'{lb} (n={len(R)})', fontsize=10)
    ax.set_xlabel('Environment steps')
axes[0].set_ylabel('IQM eval return (95% CI)')
axes[0].set_ylim(0, 520)
plt.tight_layout()
plt.show()